In [1]:
import pandas as pd

In [2]:
from pathlib import Path
base = Path("../../tasks/task_01/data/")

In [3]:
customer_df = pd.read_csv(base/ "customers.csv")
product_df = pd.read_csv(base/ "products.csv")
order_df = pd.read_csv(base/ "orders.csv")
returns_df = pd.read_csv(base/ "returns.csv")

Q1:What is the total net revenue for customers in the west region after subtracting returned units at the original discounted unit price, rounded to 2 decimals?

In [4]:
orders = order_df.merge(customer_df, on="customer_id").merge(product_df, on="product_id")
returned = returns_df.groupby("order_id", as_index=False)["returned_qty"].sum()
orders = orders.merge(returned, on="order_id", how="left").fillna({"returned_qty": 0})
orders["net_revenue"] = (orders["quantity"] - orders["returned_qty"]) * orders["unit_price"] * (1 - orders["discount_pct"] / 100)
west_net_revenue = orders.loc[orders['region'] == 'West', 'net_revenue'].sum()
print(f"{west_net_revenue:.2f}")

1521.95


Q3: On which date did total net revenue have the largest increase versus the previous date?

In [6]:
daily = orders.groupby("order_date", as_index=False)["net_revenue"].sum().sort_values("order_date")
daily["change"] = daily["net_revenue"].diff()
max_change_date = pd.Timestamp(daily.loc[daily["change"].idxmax(), "order_date"]).date().isoformat()
print(max_change_date)

2024-01-27


Q4: Which product_id has the highest return rate?

In [7]:
return_rate = (
    orders.groupby("product_id")["returned_qty"].sum()
    / orders.groupby("product_id")["quantity"].sum()
).sort_values(ascending=False)
highest_return_rate = str(return_rate.index[0])
print(highest_return_rate)


P02
